# Turbine blade defect detector — v2

Retrain after the v1 audit. Every setting below that differs from v1 is there for a reason
recorded in `docs/DATASET_AUDIT_v1.md`; the comments say which.

**Before you start.** You do **not** need to attach a Kaggle Dataset — the setup cell clones
the repo from GitHub itself, so "No input attached" in the right-hand panel is expected and
correct. Everything lands in `/kaggle/working/`, which is why the repo shows up under
**Output** rather than **Input**. (Attaching it as a Dataset also works; the cell finds it
either way.)

Two settings do matter, both under the three-dot menu: **Internet: On** (needed for the clone) and **Accelerator: GPU T4 x2**. torch 2.10 dropped Pascal (sm_60) support, so Kaggle's P100 is detected but
unusable. The notebook sets `device=0`, so only one T4 is used and DDP never engages —
which is what you want: the v1 run used `device=0,1` and the sync cost more than the second
GPU returned.

Expect roughly **2 hours for 100 epochs** on a T4: the rebuild yields ~1,832 train images,
so an epoch is ~229 iterations at ~60s. Comfortably inside Kaggle's 9-hour session limit.
The v1 run asked for 100 epochs, converged at 38, and died to a timeout at 58.

**To run it unattended**, use **Save Version -> Save & Run All (Commit)** rather than
an interactive Draft Session. A draft dies when the browser disconnects; a committed
version runs on Kaggle's infrastructure and saves its output regardless.

In [ ]:
!pip install -q ultralytics onnx onnxruntime
import torch, ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)

HOWTO = (
    "\n  1. Click 'Edit' (top right) if you are in viewer mode - the settings panel"
    "\n     only exists in the editor."
    "\n  2. Open the right-hand panel: the '<' arrow on the right edge, or the"
    "\n     three-dot menu -> Accelerator."
    "\n  3. Choose 'GPU T4 x2'."
    "\n  4. Accept the restart. It WIPES /kaggle/working, so then click 'Run All'."
)

if not torch.cuda.is_available():
    raise SystemExit(
        "\n" + "=" * 74 +
        "\nNO GPU. Stop here - the training cell will fail or take days on CPU."
        "\n" + "=" * 74 + HOWTO +
        "\n\nIf Accelerator is greyed out, Kaggle needs phone verification:"
        "\n  kaggle.com -> Settings -> Phone Verification."
        "\n\nInternet must also be On in the same menu (needed to clone the repo)."
        "\n"
    )

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
supported = torch.cuda.get_arch_list()
print("cuda:", name, f"(sm_{major}{minor})")

# torch 2.10 dropped Pascal (sm_60). Kaggle still OFFERS the P100, so this is a live
# trap: the GPU is detected, torch reports it available, and the run dies later with
# a wall of warnings instead of a clear cause.
if f"sm_{major}{minor}" not in supported:
    raise SystemExit(
        "\n" + "=" * 74 +
        f"\nGPU INCOMPATIBLE WITH THIS PYTORCH BUILD."
        "\n" + "=" * 74 +
        f"\n\n  {name} is sm_{major}{minor}."
        f"\n  This torch ({torch.__version__}) supports: {' '.join(supported)}"
        "\n\n  The P100 is Pascal (sm_60), dropped in torch 2.10. Kaggle still offers"
        "\n  it, so it is easy to pick and then fail confusingly."
        "\n\n  Use the T4 instead - Turing (sm_75), supported, and faster here anyway"
        "\n  because AMP uses its tensor cores." + HOWTO + "\n"
    )

free, total = torch.cuda.mem_get_info()
print(f"vram: {total / 1e9:.1f} GB total, {free / 1e9:.1f} GB free")
if total < 14e9:
    print("\n! Under 14 GB - drop batch to 4 in the training cell if you hit OOM.")
print("\nGPU OK.")


## 1. Rebuild the dataset

Do not train on the raw export. `rebuild_turbine.py` drops `healthy` as a class, keeps those
images as background negatives, re-splits on contiguous capture-ID blocks to remove
near-duplicate leakage, and normalises the polygon labels to boxes.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

# Locate the repo however you got it onto Kaggle. Three routes all work:
#   1. Attach it as a Kaggle Dataset  (Add Input -> Datasets -> your upload)
#   2. Kaggle can build a Dataset straight from a GitHub URL (New Dataset -> Link -> GitHub)
#   3. Nothing attached? This clones it, which needs Internet switched ON in the
#      notebook settings sidebar. Kaggle disables internet by default.
REPO_URL = "https://github.com/abyyworld/Drone-visualisation-training.git"
BRANCH   = "main"
WORK     = Path("/kaggle/working/drone-inspection")

def looks_like_repo(p: Path) -> bool:
    return (p / "tools" / "rebuild_turbine.py").exists() and (p / "train" / "images").is_dir()

inputs = sorted(Path("/kaggle/input").glob("*")) if Path("/kaggle/input").exists() else []
print("Attached inputs:", [p.name for p in inputs] or "(none)")

source = None
for entry in inputs:
    if not entry.is_dir():
        continue
    # The repo may be the input itself, or one level down if the upload was zipped.
    for candidate in [entry, *[d for d in entry.iterdir() if d.is_dir()]]:
        if looks_like_repo(candidate):
            source = candidate
            break
    if source:
        break

if source:
    print(f"Found the repo at {source}")
    if not WORK.exists():
        shutil.copytree(source, WORK)
else:
    print("No attached input looks like the repo - cloning from GitHub.")
    print("If this fails, turn Internet ON in the notebook settings sidebar.")
    if not WORK.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        REPO_URL, str(WORK)], check=True)

os.chdir(WORK)
sys.path.insert(0, str(WORK))

assert looks_like_repo(WORK), (
    f"Repo looks incomplete at {WORK}. Expected tools/ and train/images/. "
    f"Contents: {[p.name for p in WORK.iterdir()][:20]}"
)
print(f"\nOK -> {WORK}")
print("train images:", len(list((WORK / 'train' / 'images').iterdir())))

# --keep-augmented: 1,946 train images instead of 853. The deduped run produced mAP50
# 0.194 on a set that thin, so keep the extra copies until an A/B says otherwise.
# Rebuild into a leakage-free, defect-only dataset. --min-sharpness 20 drops the ~15% of
# annotated images that are smears, where a box cannot teach localisation.
!python3 tools/rebuild_turbine.py --out /kaggle/working/turbine_v2 --copy --min-sharpness 20 --keep-augmented


In [ ]:
# The rebuild exists to make these checks pass. If any fail, fix the data before training -
# training on a dataset that fails this audit is how v1 happened.
!python3 tools/audit_dataset.py /kaggle/working/turbine_v2

## 2. Train

Every hyperparameter lives in **`tools/train_turbine.py`**, not in this cell. The repo is
cloned fresh on each run, so the settings are always current - a notebook that sat on
Kaggle for a week still trains with today's config. Each non-default value carries an
inline comment saying which failure it prevents.

The run prints its full configuration before the first epoch, so the log is a record of
exactly what produced the weights.

Expect ~2 hours for 100 epochs on a T4 (~1,832 train images, 229 iterations/epoch).


In [ ]:
# The whole training configuration lives in tools/train_turbine.py, which is cloned fresh
# on every run. Nothing here goes stale, so this cell never needs updating again.
# Read that file to see every setting and why it is set that way.
import subprocess
subprocess.run([
    "python3", "tools/train_turbine.py",
    "--data", "/kaggle/working/turbine_v2/data.yaml",
    "--project", "/kaggle/working/runs", "--name", "turbine_v2",
], check=True)


## 3. Evaluate honestly

Aggregate mAP is not the number to quote. v1 scored 0.782 aggregate while being useless in
the field, because `healthy` was 62% of instances and trivially separable. v2 has no such
free class, so **expect a lower aggregate — around 0.45-0.60 — and treat that as progress.**

Judge per-class defect AP, and the visual check in section 4.

In [ ]:
import subprocess, sys
# Not `!cmd \` across lines: Kaggle's IPython transform truncates the command at
# the backslash and leaves the remaining lines as invalid Python. subprocess with an
# argument list has no such failure mode, and check=True surfaces a non-zero exit.
subprocess.run([
    "python3",
    "tools/evaluate.py",
    "/kaggle/working/runs/turbine_v2/weights/best.pt",
    "--data",
    "/kaggle/working/turbine_v2/data.yaml",
    "--split",
    "test",
    "--imgsz",
    "960",
    "--out",
    "/kaggle/working/runs/eval_v2",
], check=True)


## 4. The check that actually matters

Metrics on a split drawn from the same pool as training cannot tell you whether the model
works on a photo unlike anything it trained on. v1 never had this run against it.

Attach a folder of real turbine photos from any unrelated source (search results, a public
dataset, your own flight) and **look at the boxes**. A model that boxes sky, ground, or the
blade edge is failing in a way no in-distribution metric will reveal.

In [ ]:
import subprocess
from pathlib import Path

REALITY_CHECK = Path("/kaggle/input/turbine-reality-check")   # <- your own folder

if REALITY_CHECK.exists():
    subprocess.run([
        "python3", "tools/evaluate.py",
        "/kaggle/working/runs/turbine_v2/weights/best.pt",
        "--predict", str(REALITY_CHECK), "--imgsz", "960", "--conf", "0.25",
        "--out", "/kaggle/working/runs/reality",
    ], check=True)
else:
    print("No reality-check folder attached. Do this before trusting the model.")


## 5. Export for the web app

int8 quantisation takes the download from ~38 MB to ~10 MB but usually costs a couple of
points of mAP. Measure it, then decide.

In [ ]:
import subprocess, sys
# Not `!cmd \` across lines: Kaggle's IPython transform truncates the command at
# the backslash and leaves the remaining lines as invalid Python. subprocess with an
# argument list has no such failure mode, and check=True surfaces a non-zero exit.
subprocess.run([
    "python3",
    "tools/export_onnx.py",
    "/kaggle/working/runs/turbine_v2/weights/best.pt",
    "--name",
    "turbine",
    "--imgsz",
    "960",
], check=True)

# What did int8 quantisation cost? --device cpu because Ultralytics installs
# onnxruntime-gpu, which cannot load Kaggle's CUDA and dies on IO binding.
import subprocess, sys
# Not `!cmd \` across lines: Kaggle's IPython transform truncates the command at
# the backslash and leaves the remaining lines as invalid Python. subprocess with an
# argument list has no such failure mode, and check=True surfaces a non-zero exit.
subprocess.run([
    "python3",
    "tools/evaluate.py",
    "web/models/turbine.onnx",
    "--data",
    "/kaggle/working/turbine_v2/data.yaml",
    "--split",
    "test",
    "--imgsz",
    "960",
    "--device",
    "cpu",
    "--out",
    "/kaggle/working/runs/eval_onnx",
], check=True)

subprocess.run(["ls","-lh","web/models/"], check=True)


In [ ]:
# Download web/models/turbine.onnx and manifest.json from the notebook output,
# commit them to web/models/, and the Pages deploy picks them up automatically.
!ls -lh web/models/

## 6. Publish (optional, set-and-forget)

If a GitHub token is in **Add-ons > Secrets**, this commits the exported model and its
metrics straight to this branch, so there is nothing to download by hand when you come
back online.

The secret's label can be `GITHUB_TOKEN`, `GITHUB_KEY`, `GH_TOKEN`, `github key` or a few
other spellings - the script tries them all. For anything else, add
`"--secret-name", "your label"` to the call below.

It pushes to the working branch, **never to `main`**. GitHub Pages deploys only from
`main`, so a model landing here cannot change the live site - you merge once you have
looked at the numbers. Without a token it prints setup notes and does nothing, so the
run never fails because of it.


In [ ]:
import subprocess
subprocess.run([
    "python3", "tools/publish_results.py",
    "--name", "turbine",
    "--metrics", "/kaggle/working/runs/eval_v2",
], check=True)